In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score,RandomizedSearchCV, GridSearchCV, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression

from xgboost import XGBClassifier, XGBRegressor

### 1 Data Wrangling

#### 1.1 Understanding Data

In [60]:
df = pd.read_csv("car_insurance_claim.csv")
pd.set_option('display.max_columns', None) # is a Pandas command that instructs the library to display all columns in a DataFrame, rather than truncating the output with an ellipsis (...)
df.head()

,ID,KIDSDRIV,BIRTH,AGE,HOMEKIDS,YOJ,INCOME,PARENT1,HOME_VAL,MSTATUS,GENDER,EDUCATION,OCCUPATION,TRAVTIME,CAR_USE,BLUEBOOK,TIF,CAR_TYPE,RED_CAR,OLDCLAIM,CLM_FREQ,REVOKED,MVR_PTS,CLM_AMT,CAR_AGE,CLAIM_FLAG,URBANICITY
0,63581743,0,16MAR39,60.0,0,11.0,"$67,349",No,$0,z_No,M,PhD,Professional,14,Private,"$14,230",11,Minivan,yes,"$4,461",2,No,3,$0,18.0,0,Highly Urban/ Urban
1,132761049,0,21JAN56,43.0,0,11.0,"$91,449",No,"$257,252",z_No,M,z_High School,z_Blue Collar,22,Commercial,"$14,940",1,Minivan,yes,$0,0,No,0,$0,1.0,0,Highly Urban/ Urban
2,921317019,0,18NOV51,48.0,0,11.0,"$52,881",No,$0,z_No,M,Bachelors,Manager,26,Private,"$21,970",1,Van,yes,$0,0,No,2,$0,10.0,0,Highly Urban/ Urban
3,727598473,0,05MAR64,35.0,1,10.0,"$16,039",No,"$124,191",Yes,z_F,z_High School,Clerical,5,Private,"$4,010",4,z_SUV,no,"$38,690",2,No,3,$0,10.0,0,Highly Urban/ Urban
4,450221861,0,05JUN48,51.0,0,14.0,NaN,No,"$306,251",Yes,M,<High School,z_Blue Collar,32,Private,"$15,440",7,Minivan,yes,$0,0,No,0,$0,6.0,0,Highly Urban/ Urban


#### 1.2 Basic Data Cleaning

In [61]:
df = df.copy()

col_names = {
    'KIDSDRIV': 'num_young_drivers',
    'BIRTH': 'date_of_birth',
    'AGE': 'age',
    'HOMEKIDS': 'num_of_children',
    'YOJ': 'years_job_held_for',
    'INCOME': 'income',
    'PARENT1': 'single_parent',
    'HOME_VAL': 'value_of_home',
    'MSTATUS': 'married',
    'GENDER': 'gender',
    'EDUCATION': 'highest_education',
    'OCCUPATION': 'occupation',
    'TRAVTIME': 'commute_dist',
    'CAR_USE': 'type_of_use',
    'BLUEBOOK': 'vehicle_value',
    'TIF': 'policy_tenure',
    'CAR_TYPE': 'vehicle_type',
    'RED_CAR': 'red_vehicle',
    'OLDCLAIM': '5_year_total_claims_value',
    'CLM_FREQ': '5_year_num_of_claims',
    'REVOKED': 'licence_revoked',
    'MVR_PTS': 'license_points',
    'CLM_AMT': 'new_claim_value',
    'CAR_AGE': 'vehicle_age',
    'CLAIM_FLAG': 'is_claim',
    'URBANICITY': 'address_type'
}

df = df.rename(columns=col_names)

In [62]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10302 entries, 0 to 10301
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         10302 non-null  int64  
 1   num_young_drivers          10302 non-null  int64  
 2   date_of_birth              10302 non-null  str    
 3   age                        10295 non-null  float64
 4   num_of_children            10302 non-null  int64  
 5   years_job_held_for         9754 non-null   float64
 6   income                     9732 non-null   str    
 7   single_parent              10302 non-null  str    
 8   value_of_home              9727 non-null   str    
 9   married                    10302 non-null  str    
 10  gender                     10302 non-null  str    
 11  highest_education          10302 non-null  str    
 12  occupation                 9637 non-null   str    
 13  commute_dist               10302 non-null  int64  
 14  t

In [63]:
df.duplicated().sum()

np.int64(1)

In [64]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [65]:
df.isna().sum()

ID                             0
num_young_drivers              0
date_of_birth                  0
age                            7
num_of_children                0
years_job_held_for           548
income                       570
single_parent                  0
value_of_home                575
married                        0
gender                         0
highest_education              0
occupation                   665
commute_dist                   0
type_of_use                    0
vehicle_value                  0
policy_tenure                  0
vehicle_type                   0
red_vehicle                    0
5_year_total_claims_value      0
5_year_num_of_claims           0
licence_revoked                0
license_points                 0
new_claim_value                0
vehicle_age                  639
is_claim                       0
address_type                   0
dtype: int64

In [66]:
df.head(3)

,ID,num_young_drivers,date_of_birth,age,num_of_children,years_job_held_for,income,single_parent,value_of_home,married,gender,highest_education,occupation,commute_dist,type_of_use,vehicle_value,policy_tenure,vehicle_type,red_vehicle,5_year_total_claims_value,5_year_num_of_claims,licence_revoked,license_points,new_claim_value,vehicle_age,is_claim,address_type
0,63581743,0,16MAR39,60.0,0,11.0,"$67,349",No,$0,z_No,M,PhD,Professional,14,Private,"$14,230",11,Minivan,yes,"$4,461",2,No,3,$0,18.0,0,Highly Urban/ Urban
1,132761049,0,21JAN56,43.0,0,11.0,"$91,449",No,"$257,252",z_No,M,z_High School,z_Blue Collar,22,Commercial,"$14,940",1,Minivan,yes,$0,0,No,0,$0,1.0,0,Highly Urban/ Urban
2,921317019,0,18NOV51,48.0,0,11.0,"$52,881",No,$0,z_No,M,Bachelors,Manager,26,Private,"$21,970",1,Van,yes,$0,0,No,2,$0,10.0,0,Highly Urban/ Urban


In [67]:
currency_columns = ['income', 'value_of_home', 'vehicle_value', '5_year_total_claims_value', 'new_claim_value']

for column in currency_columns:
    print(column)
    df[column] = (
    df[column]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype("Int64")
)

income
value_of_home
vehicle_value
5_year_total_claims_value
new_claim_value


In [68]:
df[currency_columns].describe()

,income,value_of_home,vehicle_value,5_year_total_claims_value,new_claim_value
count,9731.0,9726.0,10301.0,10301.0,10301.0
mean,61568.83568,154513.740284,15660.015532,4034.369479,1511.413164
std,47458.566563,129191.840215,8429.16922,8733.476588,4725.455807
min,0.0,0.0,1500.0,0.0,0.0
25%,27583.0,0.0,9200.0,0.0,0.0
50%,53526.0,160629.0,14400.0,0.0,0.0
75%,86139.5,238251.25,20890.0,4648.0,1145.0
max,367030.0,885282.0,69740.0,57037.0,123247.0
